# 📈 Trading Signal Backtester

A Python tool to test trading strategies using historical data. Perfect for analyzing how your trading signals would have performed in the past.

## 🎯 What This Does

This notebook helps you:
1. **Upload your trading signals** - When you want to buy or sell
2. **Upload market price data** - Historical OHLC (Open, High, Low, Close) prices
3. **Run a backtest** - Simulate trades and see performance
4. **View detailed results** - Charts, metrics, and trade-by-trade breakdown

## ✨ Features

- 📁 **Easy File Upload**: Supports Excel (.xlsx, .xls) and CSV files
- 🔍 **Smart Auto-Detection**: Automatically finds columns (works with TradingView exports!)
- 📊 **Interactive Charts**: Equity curve, drawdown, and trade distribution
- 💾 **Export Results**: Download trades, metrics, and equity curve as CSV
- ⚙️ **Flexible Matching**: Adjustable time window for signal-to-price matching

## 📋 How to Use

1. **Run all cells** in order (Cell → Run All, or press Shift+Enter for each cell)
2. **Upload Signal File**: Your buy/sell signals (needs: timestamp, action, price)
3. **Upload Market Data File**: Historical prices (needs: timestamp, open, high, low, close)
4. **Adjust Settings**: Set time window if needed (default: exact match)
5. **Click "Run Backtest"**: See your strategy performance!

## 📝 File Format

The tool automatically detects columns, so these variations work:
- **Timestamp**: `time`, `timestamp`, `date`, `datetime`
- **Action**: `action`, `signal`, `side`, `type` (values: `buy`/`sell`, `long`/`short`, `b`/`s`)
- **Price**: `price`, `value`, `close`
- **OHLC**: `open`, `high`, `low`, `close`

Perfect for TradingView exports! 🚀


In [ ]:
# Import standard libraries
import pandas as pd  # For working with data tables
import numpy as np   # For math operations
import matplotlib.pyplot as plt  # For basic charts
import plotly.graph_objects as go  # For interactive charts
import ipywidgets as widgets  # For buttons and file uploads
from IPython.display import display, clear_output  # For showing results
import os
import sys

# Add parent directory to path to import backtest_logic
# Add parent directory to path to import backtest_logic
import os
import sys
from pathlib import Path

# Get the project root (parent of notebooks/)
current_dir = Path(os.getcwd())
if current_dir.name == "notebooks":
    project_root = current_dir.parent
else:
    project_root = current_dir
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


# Import shared backtest logic
from backtest_logic import DataExtractor, process_backtest

# Set chart style
plt.style.use('dark_background')
%matplotlib inline


In [ ]:
# ============================================================================
# Initialize variables and UI components
# ============================================================================

# Create variables to store our data
signals_df = None  # Will store your buy/sell signals
market_data_df = None   # Will store price data
backtest_result = None       # Will store backtest results

# Create the data extractor (imported from backtest_logic.py)
extractor = DataExtractor()

# File upload widgets (now supports Excel)
signal_upload = widgets.FileUpload(
    accept='.csv,.xlsx,.xls',
    multiple=False,
    description='Signal File'
)

market_upload = widgets.FileUpload(
    accept='.csv,.xlsx,.xls',
    multiple=False,
    description='Market Data File'
)

# Sheet selector (for Excel files)
signal_sheet = widgets.Dropdown(
    options=[],
    description='Sheet:',
    disabled=True,
    layout=widgets.Layout(width='200px')
)

market_sheet = widgets.Dropdown(
    options=[],
    description='Sheet:',
    disabled=True,
    layout=widgets.Layout(width='200px')
)

# Detection info display
signal_info = widgets.Output()
market_info = widgets.Output()

# Time window input (text input instead of slider)
time_window = widgets.IntText(
    value=0,
    min=0,
    max=3600,
    description='Time Window (seconds):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='200px')
)

# Tooltip explanation
time_window_tooltip = widgets.HTML(
    value='<span title="Time window allows signals and prices to match even if they\'re slightly apart in time. 0 = exact match only. Higher values allow matching within that many seconds.">❓</span>',
    layout=widgets.Layout(margin='0 0 0 5px')
)

# Process button
process_button = widgets.Button(
    description='🚀 Run Backtest',
    button_style='success',
    layout=widgets.Layout(width='200px', height='40px')
)

# Output area for results
output = widgets.Output()


## 📤 Step 3: Upload Your Files

Use the file upload buttons below to upload your data:

### 📊 Signal File
Your trading signals with:
- **Timestamp**: When the signal occurred
- **Action**: Buy or sell signal (`buy`, `sell`, `long`, `short`, `b`, `s`)
- **Price**: Signal price or entry price

### 📈 Market Data File
Historical price data with:
- **Timestamp**: Time of the price bar
- **Open**: Opening price
- **High**: Highest price
- **Low**: Lowest price
- **Close**: Closing price

💡 **Tip**: The tool automatically detects columns, so TradingView exports work perfectly!


In [ ]:
# Define display functions before they are used
def display_results():
    if backtest_result is None:
        return
    
    metrics = backtest_result['metrics']
    trades_df = backtest_result['trades']
    equity_curve = backtest_result['equity_curve']
    drawdown = backtest_result['drawdown']
    trade_results = backtest_result['trade_results']
    
    # Display metrics
    print("\n" + "="*60)
    print("PERFORMANCE METRICS")
    print("="*60)
    print(f"Total P&L:           ${metrics['total_pnl']:,.2f}")
    print(f"Win Rate:            {metrics['win_rate']*100:.2f}%")
    print(f"Total Trades:        {metrics['total_trades']}")
    print(f"Wins / Losses:       {metrics['total_wins']} / {metrics['total_losses']}")
    print(f"Max Drawdown:        {metrics['max_drawdown']*100:.2f}%")
    print(f"Initial Capital:     ${metrics['initial_capital']:,.2f}")
    print(f"Final Capital:       ${metrics['final_capital']:,.2f}")
    print("="*60 + "\n")
    
    # Display charts
    display_equity_curve(equity_curve)
    display_drawdown(drawdown)
    display_trade_histogram(trade_results)
    
    # Display trades table
    display_trades_table(trades_df)
    
    # Export buttons
    display_export_buttons()

def display_equity_curve(equity_curve):
    if not equity_curve:
        return
    
    df = pd.DataFrame(equity_curve)
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df['timestamp'],
        y=df['equity'],
        mode='lines',
        fill='tozeroy',
        name='Equity',
        line=dict(color='#10b981', width=2),
        fillcolor='rgba(16, 185, 129, 0.3)'
    ))
    
    fig.update_layout(
        title='Equity Curve',
        xaxis_title='Time',
        yaxis_title='Equity ($)',
        template='plotly_dark',
        height=400,
        hovermode='x unified'
    )
    
    fig.show()

def display_drawdown(drawdown):
    if not drawdown:
        return
    
    df = pd.DataFrame(drawdown)
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df['timestamp'],
        y=df['drawdown'] * 100,
        mode='lines',
        fill='tozeroy',
        name='Drawdown',
        line=dict(color='#ef4444', width=2),
        fillcolor='rgba(239, 68, 68, 0.3)'
    ))
    
    fig.update_layout(
        title='Drawdown',
        xaxis_title='Time',
        yaxis_title='Drawdown (%)',
        template='plotly_dark',
        height=400,
        hovermode='x unified'
    )
    
    fig.show()

def display_trade_histogram(trade_results):
    if not trade_results:
        return
    
    # Create histogram buckets
    bucket_size = 100
    buckets = {}
    for pnl in trade_results:
        bucket = int(pnl / bucket_size) * bucket_size
        buckets[bucket] = buckets.get(bucket, 0) + 1
    
    df = pd.DataFrame([
        {'range': f'${bucket}', 'count': count}
        for bucket, count in sorted(buckets.items())
    ])
    
    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=df['range'],
        y=df['count'],
        name='Number of Trades',
        marker_color='#8b5cf6'
    ))
    
    fig.update_layout(
        title='Trade Results Distribution',
        xaxis_title='P&L Range',
        yaxis_title='Number of Trades',
        template='plotly_dark',
        height=400
    )
    
    fig.show()

def display_trades_table(trades_df):
    if trades_df.empty:
        return
    
    print("\n" + "="*60)
    print("TRADE-BY-TRADE BREAKDOWN")
    print("="*60)
    
    # Format the dataframe for display
    display_df = trades_df.copy()
    display_df['signal_timestamp'] = pd.to_datetime(display_df['signal_timestamp']).dt.strftime('%Y-%m-%d %H:%M:%S')
    display_df['execution_timestamp'] = pd.to_datetime(display_df['execution_timestamp']).dt.strftime('%Y-%m-%d %H:%M:%S')
    display_df['signal_price'] = display_df['signal_price'].apply(lambda x: f'${x:,.2f}')
    display_df['execution_price'] = display_df['execution_price'].apply(lambda x: f'${x:,.2f}')
    display_df['quantity'] = display_df['quantity'].apply(lambda x: f'{x:.4f}')
    display_df['pnl'] = display_df['pnl'].apply(lambda x: f'${x:,.2f}')
    display_df['cumulative_pnl'] = display_df['cumulative_pnl'].apply(lambda x: f'${x:,.2f}')
    display_df['is_win'] = display_df['is_win'].apply(lambda x: 'WIN' if x else 'LOSS')
    display_df.rename(columns={
        'id': 'ID',
        'signal_timestamp': 'Signal Time',
        'action': 'Action',
        'signal_price': 'Signal Price',
        'execution_timestamp': 'Execution Time',
        'execution_price': 'Execution Price',
        'quantity': 'Quantity',
        'pnl': 'P&L',
        'cumulative_pnl': 'Cumulative P&L',
        'is_win': 'Result'
    }, inplace=True)
    
    display(display_df)
    print("="*60 + "\n")

def display_export_buttons():
    export_trades_btn = widgets.Button(description='Export Trades (CSV)')
    export_metrics_btn = widgets.Button(description='Export Metrics (CSV)')
    export_equity_btn = widgets.Button(description='Export Equity Curve (CSV)')
    
    def export_trades(b):
        if backtest_result:
            export_path = os.path.join(os.getcwd(), 'backtest_trades.csv')
            backtest_result['trades'].to_csv(export_path, index=False)
            print(f"✓ Exported trades to {export_path}")
    
    def export_metrics(b):
        if backtest_result:
            metrics = backtest_result['metrics']
            metrics_df = pd.DataFrame([{
                'Metric': 'Total P&L',
                'Value': metrics['total_pnl']
            }, {
                'Metric': 'Win Rate',
                'Value': f"{metrics['win_rate']*100:.2f}%"
            }, {
                'Metric': 'Total Wins',
                'Value': metrics['total_wins']
            }, {
                'Metric': 'Total Losses',
                'Value': metrics['total_losses']
            }, {
                'Metric': 'Max Drawdown',
                'Value': f"{metrics['max_drawdown']*100:.2f}%"
            }, {
                'Metric': 'Initial Capital',
                'Value': metrics['initial_capital']
            }, {
                'Metric': 'Final Capital',
                'Value': metrics['final_capital']
            }, {
                'Metric': 'Total Trades',
                'Value': metrics['total_trades']
            }])
            export_path = os.path.join(os.getcwd(), 'backtest_metrics.csv')
            metrics_df.to_csv(export_path, index=False)
            print(f"✓ Exported metrics to {export_path}")
    
    def export_equity(b):
        if backtest_result:
            export_path = os.path.join(os.getcwd(), 'backtest_equity_curve.csv')
            pd.DataFrame(backtest_result['equity_curve']).to_csv(export_path, index=False)
            print(f"✓ Exported equity curve to {export_path}")
    
    export_trades_btn.on_click(export_trades)
    export_metrics_btn.on_click(export_metrics)
    export_equity_btn.on_click(export_equity)
    
    print("\nExport Options:")
    display(widgets.HBox([export_trades_btn, export_metrics_btn, export_equity_btn]))


In [ ]:
def save_uploaded_file(uploaded_file, filename):
    """Save uploaded file to disk temporarily."""
    content = uploaded_file['content']
    # Save in notebooks directory
    filepath = os.path.join(os.getcwd(), filename)
    with open(filepath, 'wb') as f:
        f.write(content)
    return filepath

def analyze_file(file_upload, file_type='signal'):
    """Analyze uploaded file and extract data automatically."""
    global signals_df, market_data_df
    
    if not file_upload.value:
        return
    
    uploaded_file = list(file_upload.value.values())[0]
    filename = uploaded_file['metadata']['name']
    
    # Save file temporarily in current directory (notebooks/)
    temp_filename = f'temp_{file_type}_{filename}'
    temp_filepath = save_uploaded_file(uploaded_file, temp_filename)
    
    try:
        # Get file info
        file_info = extractor.get_file_info(temp_filepath)
        
        # Update sheet selector if Excel file
        if 'sheets' in file_info and len(file_info['sheets']) > 1:
            if file_type == 'signal':
                signal_sheet.options = file_info['sheets']
                signal_sheet.disabled = False
                signal_sheet.value = file_info['sheets'][0]
            else:
                market_sheet.options = file_info['sheets']
                market_sheet.disabled = False
                market_sheet.value = file_info['sheets'][0]
        
        # Extract data
        if file_type == 'signal':
            with signal_info:
                clear_output()
                print(f"📊 Analyzing signal file: {filename}")
                print(f"   Rows: {file_info.get('rows', 'N/A')}")
                print(f"   Columns: {', '.join(file_info.get('columns', []))}")
                
                sheet = signal_sheet.value if signal_sheet.options else None
                signals_df, detection_info = extractor.extract_signals(temp_filepath, sheet)
                
                print(f"\n✅ Auto-detected columns:")
                print(f"   Timestamp: {detection_info['timestamp_column']}")
                print(f"   Action: {detection_info['action_column']}")
                print(f"   Price: {detection_info['price_column']}")
                print(f"\n📈 Extracted {detection_info['rows_processed']} valid signals")
                if detection_info['rows_removed'] > 0:
                    print(f"   ⚠️  Removed {detection_info['rows_removed']} invalid rows")
                
        else:
            with market_info:
                clear_output()
                print(f"📊 Analyzing market data file: {filename}")
                print(f"   Rows: {file_info.get('rows', 'N/A')}")
                print(f"   Columns: {', '.join(file_info.get('columns', []))}")
                
                sheet = market_sheet.value if market_sheet.options else None
                market_data_df, detection_info = extractor.extract_market_data(temp_filepath, sheet)
                
                print(f"\n✅ Auto-detected columns:")
                print(f"   Timestamp: {detection_info['timestamp_column']}")
                print(f"   OHLC columns:")
                for ohlc_type, col in detection_info['ohlc_columns'].items():
                    print(f"     {ohlc_type.upper()}: {col}")
                print(f"\n📈 Extracted {detection_info['rows_processed']} valid data points")
                if detection_info['rows_removed'] > 0:
                    print(f"   ⚠️  Removed {detection_info['rows_removed']} invalid rows")
                
        
        # Clean up temp file
        if os.path.exists(temp_filepath):
            os.remove(temp_filepath)
            
    except Exception as e:
        if file_type == 'signal':
            with signal_info:
                clear_output()
                print(f"❌ Error processing signal file: {str(e)}")
                signals_df = None
        else:
            with market_info:
                clear_output()
                print(f"❌ Error processing market data file: {str(e)}")
                market_data_df = None
        
        # Clean up temp file
        if os.path.exists(temp_filepath):
            os.remove(temp_filepath)

def on_signal_upload(change):
    if signal_upload.value:
        analyze_file(signal_upload, 'signal')

def on_market_upload(change):
    if market_upload.value:
        analyze_file(market_upload, 'market')

def on_sheet_change(change, file_type='signal'):
    """Re-analyze when sheet selection changes."""
    if file_type == 'signal' and signal_upload.value:
        analyze_file(signal_upload, 'signal')
    elif file_type == 'market' and market_upload.value:
        analyze_file(market_upload, 'market')

signal_upload.observe(on_signal_upload, names='value')
market_upload.observe(on_market_upload, names='value')
signal_sheet.observe(lambda x: on_sheet_change(x, 'signal'), names='value')
market_sheet.observe(lambda x: on_sheet_change(x, 'market'), names='value')


In [ ]:
def run_backtest(button):
    global backtest_result
    with output:
        clear_output()
        if signals_df is None or market_data_df is None:
            print("❌ Error: Please upload and analyze both signal and market data files")
            return
        
        try:
            print("🔄 Running backtest...")
            backtest_result = process_backtest(
                signals_df,
                market_data_df,
                time_window.value
            )
            print("✅ Backtest completed successfully!\n")
            display_results()
        except Exception as e:
            print(f"❌ Error during backtest: {str(e)}")
            import traceback
            traceback.print_exc()

process_button.on_click(run_backtest)


## 🚀 Ready to Use!

The interface below provides everything you need:

1. **📁 Upload Files**: Drag & drop or click to upload your signal and market data files
2. **🔍 View Detection**: See which columns were automatically detected
3. **⚙️ Adjust Settings**: Set time window for signal-to-price matching (default: exact match)
4. **▶️ Run Backtest**: Click the button to simulate your trading strategy
5. **📊 View Results**: See performance metrics, charts, and detailed trade breakdown

### 📊 What You'll See

After running the backtest, you'll get:
- **Performance Metrics**: Total P&L, win rate, max drawdown, and more
- **Equity Curve**: Visual chart showing how your capital changed over time
- **Drawdown Chart**: See the biggest losses from peak value
- **Trade Distribution**: Histogram of your trade results
- **Trade Table**: Detailed breakdown of every trade
- **Export Options**: Download results as CSV files

**👉 Run the cell below to start!**


In [ ]:
# Display UI
print("="*70)
print("TRADING SIGNAL BACKTESTER - WITH AUTO-DETECTION")
print("="*70)
print("\n✨ Features:")
print("  • Automatic column detection from Excel/CSV files")
print("  • Supports multiple Excel sheets")
print("  • Handles various date/time formats")
print("  • Smart detection of buy/sell signals and OHLC data")
print("\n📋 Instructions:")
print("  1. Upload your signal file (Excel or CSV)")
print("  2. Upload your market data file (Excel or CSV)")
print("  3. Select sheet if Excel file has multiple sheets")
print("  4. Set the time window for signal matching")
print("  5. Click 'Run Backtest' to process\n")

display(widgets.VBox([
    widgets.HTML("<h3>📊 Signal File</h3>"),
    widgets.HBox([signal_upload, signal_sheet]),
    signal_info,
    widgets.HTML("<h3>📈 Market Data File</h3>"),
    widgets.HBox([market_upload, market_sheet]),
    market_info,
    widgets.HTML("<h3>⚙️ Settings</h3>"),
    widgets.HBox([time_window, time_window_tooltip]),
    process_button,
    output
]))
